# Step 7: QLoRA fine-tuning, Run A (rank 16, alpha 32, 1 epoch) -- Exp 4

Fine-tunes Qwen2.5-Coder-1.5B-Instruct with QLoRA (4-bit nf4) on the Step 6 training set
(`sft_train.jsonl` / `sft_val.jsonl`, prompt v2, no few-shot), loss on the SQL answer
tokens only.

**Before Run All:**
- Settings -> Accelerator: **GPU T4 x2** (only one T4 used, intentional), Internet: **On**.
- Add Input -> your uploaded `sft-training-data` dataset (zip of `sft_train.jsonl` +
  `sft_val.jsonl` from `data/processed/`).
- Add-ons -> Secrets: **`HF_TOKEN`** (write access, for pushing checkpoints/the final
  adapter to a private repo) and **`WANDB_API_KEY`**.
- Edit the single line marked `CHANGE ME` in the config cell below (your HF repo id)
  before running.

**What this notebook does:**
- Tries **Unsloth** first; if installing it would change the on-disk torch/CUDA build
  (checked via a fresh subprocess, not the already-imported module -- see the vLLM
  lesson), it aborts Unsloth cleanly and falls back to **TRL-style PEFT + bitsandbytes**
  QLoRA instead. Either way, loss masking, data loading, training loop, checkpointing
  and logging are identical.
- LoRA on every attention and MLP projection (`q/k/v/o_proj`, `gate/up/down_proj`), fp16
  (T4 has no bf16), gradient checkpointing, `group_by_length` to cut padding waste.
- Loss is computed only on the SQL answer tokens: `sft_train.jsonl` already stores
  `prompt_text` (identical structure to the Step 4+5 eval prompts) and `full_text`
  (prompt + target SQL). Both are re-tokenized here with this notebook's own tokenizer,
  and the token-count difference is how many leading tokens get masked out of the loss.
- A ~30-step smoke test measures real tokens/sec and projects the full epoch's time,
  then training continues automatically in the same run.
- Checkpoints (adapter + optimizer + scheduler + trainer state) push to a private HF
  Hub repo at `last-checkpoint` at least every ~25 minutes (and at the end). A fresh
  session auto-resumes from there if it exists.
- Logs to Weights & Biases, resuming the same run id across Kaggle sessions.
- At the end: final adapter pushed to the HF repo, loss-curve plot and run metadata
  saved to `/kaggle/working/`.

## Resuming an interrupted run
Checkpointing here goes through the HF Hub, not Kaggle's Notebook Output Files, so
resuming does **not** need the Add Input dance Step 4+5 used:
1. Just open the notebook again (new Kaggle session is fine) and **Run All**.
2. The notebook checks the HF repo for a `last-checkpoint` folder; if one exists, it
   downloads it and continues from that exact step, printing `RESUMING` and which step.
   Otherwise it prints `STARTING FRESH`.
3. The W&B run id is stored alongside the checkpoint, so resumed training logs into the
   same W&B run instead of starting a new one.


In [ ]:
# Environment check -- catch a broken environment before spending any GPU time.
import subprocess
import sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available in this session. Check Settings -> Accelerator is set to GPU T4 x2, "
        "then Run All again."
    )

print("GPU:", torch.cuda.get_device_name(0))

import transformers
print("transformers:", transformers.__version__)


def installed_torch_build():
    """What's actually on disk right now, read via a FRESH subprocess -- not the
    already-imported `torch` module in this kernel, which can't reflect a pip install
    that happened after it was first imported. This is the only reliable way to check
    whether an install changed the torch build (the exact failure mode that broke the
    Step 4+5 notebook's first vLLM attempt)."""
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
        capture_output=True, text=True,
    )
    lines = r.stdout.strip().splitlines()
    return (lines[0], lines[1]) if len(lines) >= 2 else (None, None)


TORCH_BUILD_AT_START = installed_torch_build()
print("torch build on disk, recorded for later comparison:", TORCH_BUILD_AT_START)


def safe_pip_install(args, label):
    """pip install `args`, then verify the on-disk torch build didn't change. Returns
    True if safe to proceed, False (and uninstalls) if it changed."""
    before = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args, check=True)
    after = installed_torch_build()
    if after != before:
        print(f"{label}: installing it changed the on-disk torch build ({before} -> {after}) "
              f"-- stopping this path and uninstalling.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q"] + [a for a in args if not a.startswith("-")],
                        check=False)
        return False
    return True


In [ ]:
# ---- Config -- edit HF_REPO_ID before running ----
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SEED = 0

HF_REPO_ID = "CHANGE_ME/text2sql-qlora-run-a"  # <-- CHANGE ME: your HF username/repo
WANDB_PROJECT = "text2sql-qlora"
WANDB_RUN_NAME = "run-a-rank16-alpha32-1epoch"

# LoRA (Run A)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# training set was built with this cutoff (Step 6); baseball_1 is the only db it drops
MAX_SEQ_LEN = 4096

# optimizer / schedule -- see the markdown note right after the Trainer is built for why
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.03
PER_DEVICE_BATCH = 4
GRAD_ACCUM_STEPS = 8          # effective batch size = 32
NUM_EPOCHS = 1                # Run A

SMOKE_TEST_STEPS = 30
TARGET_CHECKPOINT_INTERVAL_SEC = 25 * 60   # under the ~30 min requirement, on purpose
FIXED_VAL_SUBSET_SIZE = 200                 # out of 516 in sft_val.jsonl, to save eval time

OUTPUT_DIR = "/kaggle/working/train_out"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Kaggle secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN. Add-ons -> Secrets -> add HF_TOKEN "
                        "(a Hugging Face token with write access), then Run All again.")
try:
    WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
except Exception:
    raise RuntimeError("Missing Kaggle secret WANDB_API_KEY. Add-ons -> Secrets -> add "
                        "WANDB_API_KEY, then Run All again.")
print("secrets loaded OK")


In [ ]:
# Find the Step 6 training files under /kaggle/input (recursive search, never a fixed
# dataset slug -- see the Kaggle resumability rule in CLAUDE.md).
import glob


def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None


missing = []
paths = {}
for fname in ("sft_train.jsonl", "sft_val.jsonl"):
    p = find_input_file(fname)
    if p is None:
        missing.append(fname)
    else:
        paths[fname] = p
        print(f"found {fname} -> {p}")

if missing:
    raise FileNotFoundError(
        "Missing required input file(s) under /kaggle/input: " + ", ".join(missing) +
        ". Make sure the dataset with sft_train.jsonl and sft_val.jsonl is attached."
    )

TRAIN_PATH = paths["sft_train.jsonl"]
VAL_PATH = paths["sft_val.jsonl"]


In [ ]:
# Make sure the private HF repo exists, then check it for a checkpoint to resume from.
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)

RESUME_CKPT_DIR = None
RESUMED = False
try:
    repo_files = api.list_repo_files(repo_id=HF_REPO_ID)
    if any(f.startswith("last-checkpoint/") for f in repo_files):
        local_dir = os.path.join(OUTPUT_DIR, "resumed")
        snapshot_download(repo_id=HF_REPO_ID, local_dir=local_dir, allow_patterns="last-checkpoint/*")
        RESUME_CKPT_DIR = os.path.join(local_dir, "last-checkpoint")
        RESUMED = True
except Exception as e:
    print(f"could not check/download a previous checkpoint ({e}) -- starting fresh")

if RESUMED:
    print(f">>> RESUMING from checkpoint pushed to {HF_REPO_ID}/last-checkpoint "
          f"(local copy: {RESUME_CKPT_DIR})")
else:
    print(f">>> STARTING FRESH -- no last-checkpoint found in {HF_REPO_ID} yet")


In [ ]:
# Model + tokenizer: try Unsloth first, fall back to plain TRL-style PEFT + bitsandbytes
# QLoRA if Unsloth can't be installed cleanly (version conflict) or would touch torch.
use_unsloth = False

try:
    import importlib
    importlib.import_module("unsloth")
    print("unsloth already importable, skipping install")
    unsloth_ready = True
except ImportError:
    unsloth_ready = safe_pip_install(["unsloth"], "unsloth")

if unsloth_ready:
    try:
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=torch.float16, load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES, use_gradient_checkpointing=True, random_state=SEED,
            bias="none",
        )
        use_unsloth = True
        print("using Unsloth")
    except Exception as e:
        print(f"unsloth installed but failed to load the model ({e}) -- falling back to TRL + PEFT")

if not use_unsloth:
    for pkg, import_name in [("peft", "peft"), ("bitsandbytes", "bitsandbytes"),
                              ("trl", "trl"), ("accelerate", "accelerate")]:
        try:
            __import__(import_name)
        except ImportError:
            ok = safe_pip_install([pkg], pkg)
            if not ok:
                raise RuntimeError(f"installing {pkg} would have changed the torch build -- "
                                    f"stopping rather than risk a broken environment")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={"": 0},
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("using TRL-style PEFT + bitsandbytes (Unsloth unavailable)")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # training, not generation -- labels align left-to-right
model.config.use_cache = False  # incompatible with gradient checkpointing during training
model.print_trainable_parameters()


In [ ]:
# Load + tokenize the Step 6 training data. Loss is masked to the SQL answer tokens
# only: prompt_text and full_text are tokenized fresh with THIS notebook's tokenizer
# (not trusted from the locally-precomputed token counts), and the length difference is
# exactly how many leading tokens to mask -- see build_training_example in src/prompts.py
# for why prompt_text is guaranteed to be an exact prefix of full_text.
import json

from datasets import Dataset


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def tokenize_row(row):
    prompt_ids = tokenizer(row["prompt_text"], add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(row["full_text"], add_special_tokens=False)["input_ids"]
    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels, "length": len(full_ids)}


def build_dataset(rows):
    # coarse pre-filter on the precomputed token count (cheap -- skips re-tokenizing the
    # baseball_1 outliers entirely), then an exact filter after tokenizing here
    rows = [r for r in rows if r["full_tokens"] <= MAX_SEQ_LEN]
    examples = [tokenize_row(r) for r in rows]
    examples = [e for e in examples if len(e["input_ids"]) <= MAX_SEQ_LEN]
    return Dataset.from_list(examples)


train_rows_raw = load_jsonl(TRAIN_PATH)
val_rows_raw = load_jsonl(VAL_PATH)

train_dataset = build_dataset(train_rows_raw)
val_dataset_full = build_dataset(val_rows_raw)
val_dataset = val_dataset_full.select(range(min(FIXED_VAL_SUBSET_SIZE, len(val_dataset_full))))

print(f"train: {len(train_dataset)}/{len(train_rows_raw)} kept (<= {MAX_SEQ_LEN} tokens)")
print(f"val (fixed eval subset): {len(val_dataset)}/{len(val_dataset_full)} kept, "
      f"of {len(val_rows_raw)} total")


In [ ]:
# Custom collator: pads input_ids/attention_mask/labels to the batch's own max length
# (group_by_length below keeps similar-length examples together, so this padding waste
# stays small) and tracks real (non-pad) tokens processed, for the smoke-test throughput
# measurement.
import torch

token_counter = {"total": 0}


def collate_fn(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
        attention_mask[i, :n] = 1
        labels[i, :n] = torch.tensor(b["labels"], dtype=torch.long)
    token_counter["total"] += int(attention_mask.sum().item())
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


In [ ]:
# Weights & Biases -- resume into the same run id across Kaggle sessions if we're
# resuming training, so the loss curve doesn't fragment into separate runs.
import wandb

resume_wandb_id = None
if RESUMED:
    id_path = os.path.join(RESUME_CKPT_DIR, "wandb_run_id.txt")
    if os.path.exists(id_path):
        with open(id_path, encoding="utf-8") as f:
            resume_wandb_id = f.read().strip() or None

wandb.login(key=WANDB_API_KEY)
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, id=resume_wandb_id, resume="allow")
print("W&B run id:", wandb.run.id, "(resumed)" if resume_wandb_id else "(new)")


In [ ]:
# Trainer setup.
#
# Why these choices for Run A:
# - learning_rate=2e-4: standard for LoRA, roughly 10x a typical full-fine-tune LR --
#   the adapter starts at zero (LoRA's B matrix is zero-initialized) and only ~0.1-1% of
#   parameters are being updated, so it needs a larger step size to move meaningfully in
#   one epoch.
# - cosine schedule, warmup_ratio=0.03: a short warmup avoids a destabilizing LR spike on
#   the first few steps (when LoRA's output is still close to zero), then a smooth cosine
#   decay suits a single short run better than a step schedule with no natural restart.
# - per_device_batch=4, grad_accum=8 -> effective batch 32: a T4's 16GB has to hold the
#   4-bit base model, activations for sequences up to 4,096 tokens, and LoRA gradients;
#   batch 4 is a safe per-step memory footprint, and accumulating to an effective 32
#   keeps gradient noise reasonable without needing a bigger per-step batch.
# - group_by_length=True: without it, a batch containing one long outlier forces every
#   other sequence in that batch to pad up to it -- expensive when lengths range from
#   ~250 to ~4,096 tokens (Step 6 stats).
import time

from transformers import Trainer, TrainerCallback, TrainingArguments

effective_batch_size = PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
total_steps = max(1, -(-len(train_dataset) // effective_batch_size)) * NUM_EPOCHS
eval_steps = max(1, total_steps // 4)
print(f"effective batch size: {effective_batch_size}, total optimizer steps: {total_steps}, "
      f"eval every {eval_steps} steps")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    gradient_checkpointing=True,
    group_by_length=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="no",  # custom time-based checkpointing via the callback below
    report_to=["wandb"],
    seed=SEED,
    optim="adamw_8bit" if use_unsloth else "paged_adamw_8bit",
    remove_unused_columns=False,
)


class ThroughputAndCheckpointCallback(TrainerCallback):
    """Reports real tokens/sec + a projected full-epoch time once, after
    SMOKE_TEST_STEPS steps, without stopping training. Also pushes a full resumable
    checkpoint (adapter + optimizer + scheduler + trainer state) to the HF Hub at
    `last-checkpoint` at least every TARGET_CHECKPOINT_INTERVAL_SEC, and once more at
    the very end."""

    def __init__(self, trainer_ref, total_steps):
        self.trainer_ref = trainer_ref
        self.total_steps = total_steps
        self.train_start = None
        self.last_checkpoint_time = None
        self.smoke_test_reported = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        self.last_checkpoint_time = self.train_start

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if not self.smoke_test_reported and state.global_step >= SMOKE_TEST_STEPS:
            elapsed = now - self.train_start
            tok_per_sec = token_counter["total"] / elapsed if elapsed > 0 else 0.0
            sec_per_step = elapsed / state.global_step
            projected_sec = sec_per_step * self.total_steps
            print(f"\n[smoke test] after {state.global_step} steps ({elapsed:.0f}s measured): "
                  f"{tok_per_sec:.0f} tokens/sec, {sec_per_step:.2f}s/step")
            print(f"[smoke test] projected time for all {self.total_steps} steps "
                  f"(this epoch): ~{projected_sec / 60:.1f} min -- continuing automatically\n")
            self.smoke_test_reported = True

        if now - self.last_checkpoint_time >= TARGET_CHECKPOINT_INTERVAL_SEC:
            self._checkpoint(state.global_step)
            self.last_checkpoint_time = now

    def on_train_end(self, args, state, control, **kwargs):
        self._checkpoint(state.global_step, final=True)

    def _checkpoint(self, step, final=False):
        trainer = self.trainer_ref["trainer"]
        print(f"saving checkpoint at step {step}{' (final)' if final else ''}...")
        ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoint-latest")
        os.makedirs(ckpt_dir, exist_ok=True)
        trainer.save_model(ckpt_dir)
        trainer.state.save_to_json(os.path.join(ckpt_dir, "trainer_state.json"))
        try:
            torch.save(trainer.optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
            torch.save(trainer.lr_scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
        except Exception as e:
            print(f"  (optimizer/scheduler state not saved: {e} -- a resume would restart "
                  f"optimizer momentum, but adapter weights and step count still carry over)")
        with open(os.path.join(ckpt_dir, "wandb_run_id.txt"), "w", encoding="utf-8") as f:
            f.write(wandb.run.id if wandb.run else "")
        from huggingface_hub import upload_folder
        upload_folder(repo_id=HF_REPO_ID, folder_path=ckpt_dir, path_in_repo="last-checkpoint",
                      token=HF_TOKEN, commit_message=f"checkpoint at step {step}")
        print(f"  pushed to {HF_REPO_ID}/last-checkpoint")


In [ ]:
import time

trainer_ref = {}
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)
trainer_ref["trainer"] = trainer
trainer.add_callback(ThroughputAndCheckpointCallback(trainer_ref, total_steps))

if RESUMED:
    try:
        trainer.train(resume_from_checkpoint=RESUME_CKPT_DIR)
    except Exception as e:
        print(f"resume_from_checkpoint failed ({e}) -- starting this run fresh instead "
              f"of crashing. The adapter weights in the checkpoint are still a valid "
              f"starting point if you want to load them manually.")
        trainer.train()
else:
    trainer.train()


In [ ]:
# Final save: push the adapter to the HF repo, save loss curves + run metadata locally.
import json

import matplotlib.pyplot as plt

final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

from huggingface_hub import upload_folder
upload_folder(repo_id=HF_REPO_ID, folder_path=final_dir, path_in_repo="final_adapter",
              token=HF_TOKEN, commit_message="final adapter, run A")
print(f"pushed final adapter to {HF_REPO_ID}/final_adapter")

log_history = trainer.state.log_history
train_points = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
eval_points = [(e["step"], e["eval_loss"]) for e in log_history if "eval_loss" in e]

plt.figure(figsize=(8, 5))
if train_points:
    xs, ys = zip(*train_points)
    plt.plot(xs, ys, label="train loss")
if eval_points:
    xs, ys = zip(*eval_points)
    plt.plot(xs, ys, label="val loss", marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Run A: rank 16, alpha 32, 1 epoch")
plt.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/loss_curve_run_a.png")
print("saved /kaggle/working/loss_curve_run_a.png")

run_metadata = {
    "experiment": "Exp 4: QLoRA run A",
    "model": MODEL_NAME,
    "backend": "unsloth" if use_unsloth else "trl-style peft + bitsandbytes",
    "prompt_version": "v2",
    "lora": {"rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
              "target_modules": LORA_TARGET_MODULES},
    "max_seq_len": MAX_SEQ_LEN,
    "learning_rate": LEARNING_RATE, "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_ratio": WARMUP_RATIO, "per_device_batch": PER_DEVICE_BATCH,
    "grad_accum_steps": GRAD_ACCUM_STEPS, "effective_batch_size": effective_batch_size,
    "num_epochs": NUM_EPOCHS, "seed": SEED,
    "n_train_examples": len(train_dataset), "n_eval_examples": len(val_dataset),
    "total_steps": total_steps,
    "python_version": sys.version, "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda, "transformers_version": transformers.__version__,
    "gpu_name": torch.cuda.get_device_name(0),
    "hf_repo": HF_REPO_ID, "wandb_run_id": wandb.run.id if wandb.run else None,
    "train_log_history": log_history,
}
with open("/kaggle/working/run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print("saved /kaggle/working/run_metadata.json")
wandb.finish()


## Download from `/kaggle/working/` when done:
- `loss_curve_run_a.png`
- `run_metadata.json`

The adapter itself doesn't need downloading -- it's already pushed to
`HF_REPO_ID/final_adapter` on the Hub, which Step 8's evaluation notebook will load
directly.

If you stop partway through, nothing extra to do: the next `last-checkpoint` push already
happened within the last ~25 minutes, and opening this notebook fresh and running it again
will find it and resume.
